   
## EES Public API - Dataset Queries App Table

Creates an aggregated app-layer table from the raw EES public API data, broken down **by dataset per day**.

One row per date + dataset (id, title, version), with columns for each relevant API operation type:
- **GetMetadata** and **DownloadCsv** — sourced directly from `raw_ees_data_set_versions`
- **PostQueryExecutions** — sourced from `raw_ees_query_access`, joined to `raw_ees_data_set_versions` on `dataSetVersionId` to resolve dataset info
- **publicationTitle** — resolved via `ees_scrape_api_data_catalog` joining on `api_dataset_id`

> POST queries where the `dataSetVersionId` cannot be resolved will appear with NULL dataset fields. The data quality check cell below reports these.

**Source**: `raw_ees_data_set_versions`, `raw_ees_query_access`, `ees_scrape_api_data_catalog`

**Output**: `catalog_40_copper_statistics_services.analytics_app.ees_api_dataset_queries`

In [0]:
%sql
CREATE OR REPLACE TABLE catalog_40_copper_statistics_services.analytics_app.ees_api_dataset_queries AS

WITH dataset_version_lookup AS (
  -- Distinct mapping from dataSetVersionId → dataset info
  SELECT DISTINCT
    dataSetVersionId,
    dataSetId,
    dataSetTitle,
    dataSetVersion
  FROM catalog_40_copper_statistics_services.analytics_raw.raw_ees_data_set_versions
  WHERE dataSetVersionId IS NOT NULL
),

publication_lookup AS (
  -- Map API dataset ID → publication title via the scraped catalogue
  SELECT DISTINCT
    api_dataset_id,
    publication AS publicationTitle
  FROM catalog_40_copper_statistics_services.analytics_raw.ees_scrape_api_data_catalog
  WHERE api_dataset_id IS NOT NULL
),

get_queries AS (
  -- DownloadCsv and GetMetadata from the data_set_versions endpoint
  SELECT
    _file_date AS date,
    dataSetId,
    dataSetTitle,
    dataSetVersion,
    type,
    COUNT(*) AS query_count
  FROM catalog_40_copper_statistics_services.analytics_raw.raw_ees_data_set_versions
  WHERE type IN ('DownloadCsv', 'GetMetadata')
  GROUP BY _file_date, dataSetId, dataSetTitle, dataSetVersion, type
),

post_queries AS (
  -- POST query executions, joined to resolve dataset info
  SELECT
    qa._file_date AS date,
    lk.dataSetId,
    lk.dataSetTitle,
    lk.dataSetVersion,
    'PostQueryExecutions' AS type,
    COUNT(*) AS query_count
  FROM catalog_40_copper_statistics_services.analytics_raw.raw_ees_query_access qa
  LEFT JOIN dataset_version_lookup lk
    ON qa.dataSetVersionId = lk.dataSetVersionId
  GROUP BY qa._file_date, lk.dataSetId, lk.dataSetTitle, lk.dataSetVersion
),

combined AS (
  SELECT * FROM get_queries
  UNION ALL
  SELECT * FROM post_queries
),

pivoted AS (
  SELECT *
  FROM combined
  PIVOT (
    SUM(query_count) FOR type IN (
      'DownloadCsv',
      'GetMetadata',
      'PostQueryExecutions'
    )
  )
)

SELECT
  p.date,
  p.dataSetId,
  p.dataSetTitle,
  p.dataSetVersion,
  pl.publicationTitle,
  COALESCE(p.DownloadCsv, 0)          AS DownloadCsv,
  COALESCE(p.GetMetadata, 0)          AS GetMetadata,
  COALESCE(p.PostQueryExecutions, 0)  AS PostQueryExecutions
FROM pivoted p
LEFT JOIN publication_lookup pl
  ON p.dataSetId = pl.api_dataset_id
ORDER BY p.date, p.dataSetTitle, p.dataSetVersion;

In [0]:
%sql
-- POST queries where dataSetVersionId could not be resolved to a dataset
SELECT
  date,
  SUM(PostQueryExecutions) AS unresolved_post_queries
FROM catalog_40_copper_statistics_services.analytics_app.ees_api_dataset_queries
WHERE dataSetId IS NULL
  AND PostQueryExecutions > 0
GROUP BY date
ORDER BY date;